In [1]:
import os
import pandas as pd
import numpy as np

file_name = "Telco_Customer_Churn_Dataset  (3).csv"

try:
    df = pd.read_csv(file_name)
except Exception as e:
    desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
    df = pd.read_csv(os.path.join(desktop_path, file_name))

df['Churn'] = df['Churn'].apply(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)
df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')
df['tenure'] = pd.to_numeric(df['tenure'], errors='coerce')

print("--- TASK 3: CUSTOMER SEGMENTATION ---")

def segment_customers(row):
    if row['tenure'] <= 12:
        return 'New Customers'
    elif row['tenure'] > 12 and row['MonthlyCharges'] > 70:
        return 'High-Value Loyal'
    else:
        return 'Standard Long-Term'

df['Customer_Segment'] = df.apply(segment_customers, axis=1)

segment_analysis = df.groupby('Customer_Segment').agg(
    Total_Customers=('customerID', 'count'),
    Average_Monthly_Spend=('MonthlyCharges', 'mean'),
    Churn_Rate_Percentage=('Churn', lambda x: x.mean() * 100)
).reset_index()

print("\n--- Segment Analysis Summary ---")
print(segment_analysis.to_string(index=False))

high_value_at_risk = df[
    (df['Customer_Segment'] == 'High-Value Loyal') & 
    (df['Contract'] == 'Month-to-month') & 
    (df['Churn'] == 1)
]

print(f"\n--- High-Value Customers At Risk Identified: {high_value_at_risk.shape[0]} ---")
print("Sample list of high-paying customers who left/are leaving:")
print(high_value_at_risk[['customerID', 'tenure', 'MonthlyCharges', 'Contract']].head(10).to_string(index=False))

--- TASK 3: CUSTOMER SEGMENTATION ---

--- Segment Analysis Summary ---
  Customer_Segment  Total_Customers  Average_Monthly_Spend  Churn_Rate_Percentage
  High-Value Loyal             2708              92.543852              24.852290
     New Customers             2186              56.097781              47.438243
Standard Long-Term             2149              38.565891               7.398790

--- High-Value Customers At Risk Identified: 514 ---
Sample list of high-paying customers who left/are leaving:
customerID  tenure  MonthlyCharges       Contract
7892-POOKP      28          104.80 Month-to-month
0280-XJGEX      49          103.70 Month-to-month
6467-CHFZW      47           99.35 Month-to-month
5380-WJKOV      34          106.35 Month-to-month
9420-LOJKX      15           99.10 Month-to-month
1658-BYGOY      18           95.45 Month-to-month
4598-XLKNJ      25           98.50 Month-to-month
0486-HECZI      55           96.75 Month-to-month
4846-WHAFZ      37           76.50 Mo